In [ ]:
from kickstart import load_data

dataset = load_data("mlm_items_train.jsonlines")

In [ ]:
print dataset[0].keys()

In [ ]:
print dataset[0].values()

In [ ]:
from random import shuffle

shuffle(dataset)
print dataset[0].values()

Genera el dataset en formato DataFrame

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

Separo los labels (y)

In [ ]:
y = df['top_level_category']

columns = [u'seller_id', u'price', 'title']
X = df[columns]

Split en train y valid (test) sets

In [ ]:
from sklearn.cross_validation import StratifiedKFold
# ...
eval_size = .1 

kf = StratifiedKFold(y, round(1. / eval_size))
train_indices, valid_indices = next(iter(kf))

In [ ]:
X_train, y_train = X.ix[train_indices], y[train_indices]
X_valid, y_valid = X.ix[valid_indices], y[valid_indices]

In [ ]:
X_train.head()

In [ ]:
X_train['title'] = X_train['title'].str.lower().str.strip().str.replace(',', ' ').str.replace('.', ' ')
X_train.head()

In [ ]:
import re

STOPWORDS = [
    'para', 'de', 'y', 'con', 'la', 'las', 'los', 'el', 'envio', 'gratis',
    'o', 'por', 'en', '-', '!', '\\', '/', 'a', 'un', 'una', 'unos', 'unas',
    'del', '+', 'tipo'
]

def remove_words(input_text):
    resultwords  = [word for word in input_text.split() if word.lower() not in STOPWORDS]
    result = re.sub(" \d+", "", ' '.join(resultwords))
    return result

In [ ]:
X_train['title'] = X_train['title'].apply(remove_words)
X_train.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
ctv = CountVectorizer(strip_accents='ascii', stop_words=STOPWORDS, max_df=0.7)

X_train_title_ctv = ctv.fit_transform(X_train['title'])

In [ ]:
X_train_title_ctv.shape

In [ ]:
len(ctv.stop_words_)

In [ ]:
X_train_title_ctv

In [ ]:
X_train_title_ctv_df = pd.DataFrame(X_train_title_ctv.A, columns=ctv.get_feature_names()) 

In [ ]:
from sklearn.linear_model import SGDClassifier

clf = SGDClassifier(loss='log', n_jobs=3)
clf.fit(X_train, y_train)

In [ ]:
pred = clf.predict(X_valid)

In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_valid, pred)
print "Accuracy of estimator: %0.2f" %acc

In [ ]:
clf2 = SGDClassifier(loss='hinge', n_jobs=3)
clf2.fit(X_train, y_train)
pred = clf2.predict(X_valid)

acc = accuracy_score(y_valid, pred)
print "Accuracy of estimator: %0.2f" %acc

Voy a probar XGBoost a ver que resultado me da...

In [ ]:
import xgboost as xgb

gbm = xgb.XGBClassifier(n_estimators=15).fit(X_train, y_train)
pred = gbm.predict(X_valid)
print "Accuracy of XGBoost: %0.2f" %accuracy_score(y_valid, pred)

Identificar y separar diferentes tipos de variables en los datos (numerical, categorical, text).

In [ ]:
from sklearn.preprocessing import LabelEncoder

lbl_enc = LabelEncoder()
lbl_enc.fit(X_train[u'seller_power_status'])
X_train_cat_sps = lbl_enc.transform(X_train[u'seller_power_status'])

In [ ]:
X_train_cat_sps

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()
ohe.fit(X_train['seller_id'].values.reshape(-1, 1))
X_train_enc = ohe.transform(X_train['seller_id'].values.reshape(-1, 1))

In [ ]:
X_train_enc

In [ ]:
len(set(X_train['seller_id']))

Formatear con countVectorizer los titulos.

In [ ]:
text_data_train = list(X_train.apply(lambda x: '%s' %(x[u'title']), axis=1))
text_data_valid = list(X_valid.apply(lambda x: '%s' %(x[u'title']), axis=1))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

ctv = CountVectorizer()
tdt_ctv = ctv.fit_transform(text_data_train)
tdv_ctv = ctv.fit_transform(text_data_valid)

In [ ]:
tdt_ctv

In [ ]:
X_train.groupby('id')

Filtrar y remover features que no voy a necesitar.

In [ ]:
import pandas as pd
import numpy as np

columns = [u'seller_id', u'top_level_category', u'price']
df = pd.DataFrame(dataset)
df.drop([label for label in dataset[0].keys() if label not in columns], axis=1, inplace=True)

df.seller_id = df.seller_id.astype(str)

df.ix[0]

Transformamos a una lista de diccionarios para poder vectorizar.

In [ ]:
df_d = df.T.to_dict().values()
df_d[0]

Ahora vectorizamos el set.

In [ ]:
from sklearn.feature_extraction import DictVectorizer

vec = DictVectorizer()
df_dv = vec.fit_transform(df_d)

df_dv[0]

In [ ]:
from collections import Counter

for_sale = Counter(x['seller_id'] for x in train)
vectorize = lambda datum: (datum['price'], for_sale[datum['seller_id']])

In [ ]:
train_len = int(len(train)*0.9)
train_split = train[:train_len]
test_split = train[train_len:]

print len(train_split)
print len(test_split)

Generar el train set de inputs.

In [ ]:
X_train = [vectorize(x) for x in train_split]

Generar el train set de outputs.

In [ ]:
#from sklearn.preprocessing import LabelEncoder

#enc = LabelEncoder()
#y_train = enc.fit_transform(pd.DataFrame(train_split)['top_level_category'].tolist())

y_train = [x["top_level_category"] for x in train_split]

De igual manera con los tests sets.

In [ ]:
#X_test = pd.DataFrame(test_split).as_matrix(columns).astype(np.float)
#y_test = enc.fit_transform(pd.DataFrame(test_split)['top_level_category'].tolist())

X_test = [vectorize(x) for x in test_split]
y_test = [x["top_level_category"] for x in test_split]

Entrenamos el clasificador elegido.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

train_enc = OneHotEncoder()
y_train = train_enc.fit_transform(y_train)

test_enc = OneHotEncoder()
y_test = test_enc.fit_transform(y_test)

In [ ]:
y_train[0]
y_test[0]

In [ ]:
#from sklearn.tree import DecisionTreeClassifier
#classifier = DecisionTreeClassifier()

from sklearn.linear_model import LinearRegression
classifier = LinearRegression(n_jobs=2)

In [ ]:
classifier.fit(X_train, y_train)

In [ ]:
prediction = classifier.predict(X_test)

from random import randint

nidx = randint(0, len(prediction))
print enc.inverse_transform([int(prediction[nidx])])
print enc.inverse_transform([y_test[nidx]])

In [ ]:
sames = sum(1 for y1, y2 in zip(y_test, prediction) 
            if enc.inverse_transform([int(y1)]) == enc.inverse_transform([int(y2)]))
print sames*1./len(y_test)

In [ ]:
def calc_score(y_test, prediction):
    sames = sum(1 for y1, y2 in zip(y_test, prediction) if y1 == y2)
    return sames*1./len(y_test)

In [ ]:
print calc_score(y_test, prediction)

Vamos a probar cross val score, para variar el slice de test. KFold cross val. Por ejemplo si cortamos en slices de 10% para validar, es 10-fold cv.

In [ ]:
from sklearn.cross_validation import cross_val_score

scores = cross_val_score(classifier, X_train, y=y_train, cv=10, n_jobs=2)
print scores.mean()